# SpyCloud Threat Hunting Notebook

**Purpose:** Provide SOC analysts with structured threat hunts that leverage SpyCloud breach and malware exposure data
ingested into Microsoft Sentinel. Each hunt is self-contained with a description, KQL query, and visualization.

| Item | Detail |
|------|--------|
| **Data source** | `SpyCloudBreachWatchlist_CL` custom table |
| **Correlation tables** | `SigninLogs`, `OfficeActivity`, `DeviceNetworkEvents`, `IdentityInfo` |
| **Minimum role** | Microsoft Sentinel Reader + Log Analytics Reader |
| **Estimated runtime** | 15–25 minutes for all hunts on a 90-day dataset |

---

## Setup & Authentication

Connect to your Log Analytics workspace using `msticpy`. Ensure you have the
`msticpy`, `pandas`, `matplotlib`, and `plotly` packages installed.

Update `TENANT_ID` and `WORKSPACE_ID` below, or configure
`msticpyconfig.yaml` for persistent settings.

In [ ]:
# Install dependencies (uncomment on first run)
# %pip install msticpy[all] pandas matplotlib plotly azure-identity

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
from IPython.display import display, HTML, Markdown

import msticpy as mp
mp.init_notebook(namespace=globals(), verbosity=0)

print("Libraries loaded successfully.")

In [ ]:
# === Workspace Configuration ===
TENANT_ID    = "YOUR_TENANT_ID"      # Azure AD / Entra ID tenant
WORKSPACE_ID = "YOUR_WORKSPACE_ID"    # Log Analytics workspace ID
LOOKBACK     = 30                      # Default lookback in days

# Connect to Sentinel
qry_prov = QueryProvider("MSSentinel")
qry_prov.connect(
    connection_str=(
        f"loganalytics://code().tenant('{TENANT_ID}')"
        f".workspace('{WORKSPACE_ID}')"
    )
)
print(f"Connected to workspace {WORKSPACE_ID}")
print(f"Default lookback: {LOOKBACK} days")

In [ ]:
# Helper: run a KQL query and return a DataFrame
def run_query(kql: str) -> pd.DataFrame:
    """Execute KQL against the connected workspace and return results as a DataFrame."""
    df = qry_prov.exec_query(kql)
    if df is None or df.empty:
        print("  [!] Query returned no results.")
        return pd.DataFrame()
    print(f"  [+] {len(df)} rows returned.")
    return df


def severity_color(sev):
    """Map SpyCloud severity (1-25) to a color for charts."""
    if sev >= 20:
        return "#d32f2f"   # critical - red
    elif sev >= 15:
        return "#f57c00"   # high - orange
    elif sev >= 10:
        return "#fbc02d"   # medium - yellow
    elif sev >= 5:
        return "#1976d2"   # low - blue
    return "#9e9e9e"       # info - gray


print("Helper functions ready.")

---
## Hunt 1: Compromised Executive Accounts

**Objective:** Identify C-suite and VIP users whose credentials appear in SpyCloud breach data.
Executive accounts carry elevated privileges and access to sensitive strategy, financial, and M&A data.
A single compromised executive credential can lead to BEC fraud, data exfiltration, or full-environment takeover.

**MITRE ATT&CK:** T1078 (Valid Accounts), T1589.001 (Gather Victim Identity — Credentials)

**Data sources:** `SpyCloudBreachWatchlist_CL`, `SpyCloudVIP` watchlist

In [ ]:
# Hunt 1 - KQL: Compromised Executive Accounts
hunt1_kql = f"""
let vip_users = _GetWatchlist('SpyCloudVIP')
| project Email=tolower(Email), Name, Title, Priority;
SpyCloudBreachWatchlist_CL
| where TimeGenerated > ago({LOOKBACK}d)
| extend email_lower = tolower(email)
| join kind=inner vip_users on $left.email_lower == $right.Email
| summarize
    Exposures        = count(),
    MaxSeverity      = max(severity),
    PlaintextPasswords = countif(isnotempty(password_plaintext)),
    BreachDomains    = dcount(target_domain),
    DomainList       = make_set(target_domain, 10),
    InfectedDevices  = dcount(infected_machine_id),
    LatestExposure   = max(spycloud_publish_date)
    by email, Name, Title, Priority
| extend DaysSinceExposure = datetime_diff('day', now(), LatestExposure)
| order by Priority asc, MaxSeverity desc
"""

df_hunt1 = run_query(hunt1_kql)
display(df_hunt1)

In [ ]:
# Hunt 1 - Visualization: Executive Exposure Heatmap
if not df_hunt1.empty:
    fig = px.bar(
        df_hunt1.sort_values("MaxSeverity", ascending=True),
        y="Name",
        x="Exposures",
        color="MaxSeverity",
        color_continuous_scale="RdYlGn_r",
        orientation="h",
        title="Hunt 1: Executive Credential Exposures by Severity",
        labels={"Exposures": "Total Exposures", "MaxSeverity": "Severity"},
        hover_data=["Title", "PlaintextPasswords", "BreachDomains", "DaysSinceExposure"]
    )
    fig.update_layout(height=max(400, len(df_hunt1) * 40), yaxis_title="")
    fig.show()
else:
    print("No executive exposures found. Ensure the SpyCloudVIP watchlist is populated.")

---
## Hunt 2: Credential Stuffing Detection

**Objective:** Correlate SpyCloud breach records with Entra ID failed sign-in events to detect
active credential-stuffing campaigns. A spike in failed logins for accounts with known-exposed
credentials is a strong signal that attackers are weaponizing breach data in real time.

**MITRE ATT&CK:** T1110.004 (Credential Stuffing)

**Data sources:** `SpyCloudBreachWatchlist_CL`, `SigninLogs`

In [ ]:
# Hunt 2 - KQL: Credential Stuffing Detection
hunt2_kql = f"""
let exposed_creds = SpyCloudBreachWatchlist_CL
| where TimeGenerated > ago({LOOKBACK}d)
| where severity >= 5
| summarize
    MaxSev       = max(severity),
    Exposures    = count(),
    HasPlaintext = countif(isnotempty(password_plaintext))
    by email;
SigninLogs
| where TimeGenerated > ago(7d)
| where ResultType != 0  // failed sign-ins only
| where UserPrincipalName in~ (exposed_creds | project email)
| summarize
    FailedAttempts   = count(),
    DistinctIPs      = dcount(IPAddress),
    IPList           = make_set(IPAddress, 20),
    DistinctApps     = dcount(AppDisplayName),
    ErrorCodes       = make_set(ResultType, 10),
    Locations        = make_set(Location, 10),
    FirstAttempt     = min(TimeGenerated),
    LastAttempt      = max(TimeGenerated)
    by UserPrincipalName
| join kind=inner exposed_creds on $left.UserPrincipalName == $right.email
| extend AttackWindow_Hours = datetime_diff('hour', LastAttempt, FirstAttempt)
| where FailedAttempts >= 5
| order by FailedAttempts desc, MaxSev desc
"""

df_hunt2 = run_query(hunt2_kql)
display(df_hunt2)

In [ ]:
# Hunt 2 - Visualization: Failed sign-in volume for exposed accounts
if not df_hunt2.empty:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Failed Attempts per User", "Distinct Source IPs per User"),
        horizontal_spacing=0.15
    )
    top_n = df_hunt2.head(15)
    fig.add_trace(
        go.Bar(x=top_n["UserPrincipalName"], y=top_n["FailedAttempts"],
               name="Failed Attempts", marker_color="#d32f2f"),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(x=top_n["UserPrincipalName"], y=top_n["DistinctIPs"],
               name="Distinct IPs", marker_color="#1976d2"),
        row=1, col=2
    )
    fig.update_layout(
        title_text="Hunt 2: Credential Stuffing Indicators",
        height=500, showlegend=True
    )
    fig.update_xaxes(tickangle=45)
    fig.show()
else:
    print("No credential stuffing patterns detected in the lookback window.")

---
## Hunt 3: Lateral Movement via Stolen Credentials

**Objective:** Detect whether credentials stolen by infostealers and surfaced by SpyCloud are
being used to authenticate across multiple internal systems. Successful sign-ins from exposed
accounts to different applications, hosts, or services within a short window indicate potential
lateral movement.

**MITRE ATT&CK:** T1021 (Remote Services), T1078 (Valid Accounts)

**Data sources:** `SpyCloudBreachWatchlist_CL`, `SigninLogs`, `DeviceLogonEvents`

In [ ]:
# Hunt 3 - KQL: Lateral Movement via Stolen Credentials
hunt3_kql = f"""
let stolen_creds = SpyCloudBreachWatchlist_CL
| where TimeGenerated > ago({LOOKBACK}d)
| where severity >= 10
| where isnotempty(password_plaintext) or isnotempty(password)
| summarize MaxSev = max(severity), Exposures = count() by email;
// Successful sign-ins across multiple apps from exposed users
let multi_app_logins = SigninLogs
| where TimeGenerated > ago(7d)
| where ResultType == 0
| where UserPrincipalName in~ (stolen_creds | project email)
| summarize
    SuccessfulLogins = count(),
    DistinctApps     = dcount(AppDisplayName),
    AppList          = make_set(AppDisplayName, 15),
    DistinctIPs      = dcount(IPAddress),
    IPList           = make_set(IPAddress, 10),
    Locations        = make_set(Location, 10),
    FirstLogin       = min(TimeGenerated),
    LastLogin        = max(TimeGenerated)
    by UserPrincipalName
| where DistinctApps >= 3;
multi_app_logins
| join kind=inner stolen_creds on $left.UserPrincipalName == $right.email
| extend SpreadWindow_Hours = datetime_diff('hour', LastLogin, FirstLogin)
| project UserPrincipalName, MaxSev, Exposures, SuccessfulLogins,
          DistinctApps, AppList, DistinctIPs, IPList, Locations,
          SpreadWindow_Hours
| order by DistinctApps desc, MaxSev desc
"""

df_hunt3 = run_query(hunt3_kql)
display(df_hunt3)

In [ ]:
# Hunt 3 - Visualization: Lateral movement spread map
if not df_hunt3.empty:
    fig = px.scatter(
        df_hunt3,
        x="DistinctApps",
        y="DistinctIPs",
        size="SuccessfulLogins",
        color="MaxSev",
        color_continuous_scale="RdYlGn_r",
        hover_name="UserPrincipalName",
        hover_data=["SpreadWindow_Hours", "Exposures"],
        title="Hunt 3: Lateral Movement — App Spread vs Source IPs (size = login count)",
        labels={
            "DistinctApps": "Distinct Applications Accessed",
            "DistinctIPs": "Distinct Source IPs",
            "MaxSev": "Exposure Severity"
        }
    )
    fig.update_layout(height=550)
    fig.show()
else:
    print("No lateral movement patterns detected.")

---
## Hunt 4: Dark Web Exposure Timeline

**Objective:** Visualize when organizational credentials first appeared in dark web breach
compilations and infostealer logs. Understanding the temporal distribution of exposures helps
identify whether the organization was hit by a single large breach or faces ongoing, persistent
credential theft.

**MITRE ATT&CK:** T1589.001 (Gather Victim Identity — Credentials)

**Data sources:** `SpyCloudBreachWatchlist_CL`

In [ ]:
# Hunt 4 - KQL: Dark Web Exposure Timeline
hunt4_kql = f"""
SpyCloudBreachWatchlist_CL
| where TimeGenerated > ago(365d)
| where isnotempty(spycloud_publish_date)
| extend PublishDate = todatetime(spycloud_publish_date)
| summarize
    TotalExposures   = count(),
    UniqueEmails     = dcount(email),
    PlaintextCount   = countif(isnotempty(password_plaintext)),
    AvgSeverity      = avg(severity),
    MaxSeverity      = max(severity),
    BreachSources    = dcount(source_id)
    by PublishWeek = startofweek(PublishDate)
| order by PublishWeek asc
"""

df_hunt4 = run_query(hunt4_kql)
display(df_hunt4)

In [ ]:
# Hunt 4 - Visualization: Exposure timeline with severity overlay
if not df_hunt4.empty:
    df_hunt4["PublishWeek"] = pd.to_datetime(df_hunt4["PublishWeek"])

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(
        go.Bar(
            x=df_hunt4["PublishWeek"],
            y=df_hunt4["TotalExposures"],
            name="Total Exposures",
            marker_color="#1976d2",
            opacity=0.7
        ),
        secondary_y=False
    )

    fig.add_trace(
        go.Scatter(
            x=df_hunt4["PublishWeek"],
            y=df_hunt4["MaxSeverity"],
            name="Max Severity",
            line=dict(color="#d32f2f", width=2),
            mode="lines+markers"
        ),
        secondary_y=True
    )

    fig.add_trace(
        go.Scatter(
            x=df_hunt4["PublishWeek"],
            y=df_hunt4["UniqueEmails"],
            name="Unique Emails",
            line=dict(color="#388e3c", width=2, dash="dash"),
            mode="lines+markers"
        ),
        secondary_y=False
    )

    fig.update_layout(
        title_text="Hunt 4: Dark Web Credential Exposure Timeline",
        height=500,
        xaxis_title="Breach Publish Week"
    )
    fig.update_yaxes(title_text="Count", secondary_y=False)
    fig.update_yaxes(title_text="Severity (0–25)", secondary_y=True)
    fig.show()
else:
    print("No timeline data available.")

---
## Hunt 5: Third-Party Risk Assessment

**Objective:** Identify breaches originating from partner, vendor, or third-party service domains.
Employees frequently reuse corporate credentials on external platforms. When those platforms are
breached, the credentials become an entry vector into **your** environment.

**MITRE ATT&CK:** T1078.004 (Cloud Accounts), T1199 (Trusted Relationship)

**Data sources:** `SpyCloudBreachWatchlist_CL`

In [ ]:
# Hunt 5 - KQL: Third-Party Risk Assessment
# Customize CORPORATE_DOMAIN to your organization's primary email domain
CORPORATE_DOMAIN = "yourcompany.com"

hunt5_kql = f"""
SpyCloudBreachWatchlist_CL
| where TimeGenerated > ago({LOOKBACK}d)
| where email_domain =~ "{CORPORATE_DOMAIN}"
| where target_domain !~ "{CORPORATE_DOMAIN}"  // external sites only
| where isnotempty(target_domain)
| summarize
    ExposedUsers     = dcount(email),
    TotalExposures   = count(),
    MaxSeverity      = max(severity),
    PlaintextPWs     = countif(isnotempty(password_plaintext)),
    SampleUsers      = make_set(email, 5),
    LatestExposure   = max(spycloud_publish_date)
    by target_domain
| extend RiskScore = ExposedUsers * 2 + PlaintextPWs * 5 + MaxSeverity
| order by RiskScore desc
| limit 50
"""

df_hunt5 = run_query(hunt5_kql)
display(df_hunt5)

In [ ]:
# Hunt 5 - Visualization: Third-party risk treemap
if not df_hunt5.empty:
    fig = px.treemap(
        df_hunt5.head(30),
        path=["target_domain"],
        values="ExposedUsers",
        color="MaxSeverity",
        color_continuous_scale="RdYlGn_r",
        title="Hunt 5: Third-Party Breach Domains — Size = Exposed Users, Color = Severity",
        hover_data=["TotalExposures", "PlaintextPWs", "RiskScore"]
    )
    fig.update_layout(height=600)
    fig.show()
else:
    print("No third-party exposures found. Verify CORPORATE_DOMAIN is correct.")

---
## Hunt 6: Password Reuse Detection

**Objective:** Find users whose same password hash appears across multiple breached domains.
Password reuse is the number one enabler of credential stuffing. If a user reuses their
corporate password on external sites, a breach of **any** of those sites compromises the
corporate account.

**MITRE ATT&CK:** T1110.004 (Credential Stuffing), T1552 (Unsecured Credentials)

**Data sources:** `SpyCloudBreachWatchlist_CL`

In [ ]:
# Hunt 6 - KQL: Password Reuse Detection
hunt6_kql = f"""
SpyCloudBreachWatchlist_CL
| where TimeGenerated > ago({LOOKBACK}d)
| where isnotempty(password)
| summarize
    DomainsWithSamePW = dcount(target_domain),
    DomainList        = make_set(target_domain, 20),
    Exposures         = count(),
    MaxSeverity       = max(severity),
    HasPlaintext      = countif(isnotempty(password_plaintext)),
    PasswordTypes     = make_set(password_type, 5),
    LatestExposure    = max(spycloud_publish_date)
    by email, password_hash = hash_sha256(password)
| where DomainsWithSamePW >= 2
| project-away password_hash
| extend DaysSinceExposure = datetime_diff('day', now(), todatetime(LatestExposure))
| order by DomainsWithSamePW desc, MaxSeverity desc
"""

df_hunt6 = run_query(hunt6_kql)
display(df_hunt6)

In [ ]:
# Hunt 6 - Visualization: Password reuse severity distribution
if not df_hunt6.empty:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "Domains with Reused Password per User",
            "Reuse Count Distribution"
        )
    )

    top_reusers = df_hunt6.head(20)
    fig.add_trace(
        go.Bar(
            x=top_reusers["email"],
            y=top_reusers["DomainsWithSamePW"],
            marker_color="#e65100",
            name="Domains"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Histogram(
            x=df_hunt6["DomainsWithSamePW"],
            marker_color="#1976d2",
            name="User Count"
        ),
        row=1, col=2
    )

    fig.update_layout(
        title_text="Hunt 6: Password Reuse Analysis",
        height=500, showlegend=True
    )
    fig.update_xaxes(tickangle=45, row=1, col=1)
    fig.show()
else:
    print("No password reuse detected in the lookback window.")

---
## Hunt 7: Botnet Infection Indicators

**Objective:** Identify credentials that were stolen by infostealer malware (botnets) rather than
traditional site breaches. Infostealer-harvested credentials are the most dangerous exposure type
because they are recent, accurate, and often include session cookies that bypass MFA.

**MITRE ATT&CK:** T1539 (Steal Web Session Cookie), T1555 (Credentials from Password Stores)

**Data sources:** `SpyCloudBreachWatchlist_CL`

In [ ]:
# Hunt 7 - KQL: Botnet Infection Indicators
hunt7_kql = f"""
SpyCloudBreachWatchlist_CL
| where TimeGenerated > ago({LOOKBACK}d)
| where isnotempty(infected_machine_id) or isnotempty(infected_time)
| summarize
    InfectionRecords  = count(),
    MaxSeverity       = max(severity),
    StealerFamilies   = make_set(password_type, 10),
    InfectedDevices   = dcount(infected_machine_id),
    DeviceNames       = make_set(user_hostname, 10),
    OperatingSystems  = make_set(user_os, 5),
    AffectedDomains   = dcount(target_domain),
    DomainList        = make_set(target_domain, 15),
    CookieCount       = countif(password_type has_any ('cookie', 'session', 'token')),
    FirstInfection    = min(infected_time),
    LastInfection     = max(infected_time)
    by email
| extend InfectionSpan_Days = datetime_diff('day', LastInfection, FirstInfection)
| extend HasSessionTokens = CookieCount > 0
| order by MaxSeverity desc, InfectionRecords desc
"""

df_hunt7 = run_query(hunt7_kql)
display(df_hunt7)

In [ ]:
# Hunt 7 - Visualization: Botnet infection summary
if not df_hunt7.empty:
    fig = px.scatter(
        df_hunt7,
        x="InfectedDevices",
        y="AffectedDomains",
        size="InfectionRecords",
        color="HasSessionTokens",
        color_discrete_map={True: "#d32f2f", False: "#1976d2"},
        hover_name="email",
        hover_data=["MaxSeverity", "InfectionSpan_Days", "CookieCount"],
        title="Hunt 7: Botnet Infections — Devices vs Domains (red = has stolen session tokens)",
        labels={
            "InfectedDevices": "Infected Devices",
            "AffectedDomains": "Affected Domains",
            "HasSessionTokens": "Session Tokens Stolen"
        }
    )
    fig.update_layout(height=550)
    fig.show()

    # Summary statistics
    total_infected = df_hunt7["InfectedDevices"].sum()
    token_users = df_hunt7["HasSessionTokens"].sum()
    display(Markdown(
        f"**Botnet Summary:** {len(df_hunt7)} users affected, "
        f"{total_infected} infected devices, "
        f"{token_users} users with stolen session tokens (MFA bypass risk)"
    ))
else:
    print("No botnet infection indicators found.")

---
## Hunt 8: Geographic Anomaly Detection

**Objective:** Compare the geographic origin of SpyCloud infostealer infections and breach
data against normal user sign-in locations. A user whose credentials were stolen from a
country where they never sign in from warrants immediate investigation.

**MITRE ATT&CK:** T1078 (Valid Accounts), T1071 (Application Layer Protocol)

**Data sources:** `SpyCloudBreachWatchlist_CL`, `SigninLogs`

In [ ]:
# Hunt 8 - KQL: Geographic Anomaly Detection
hunt8_kql = f"""
// Step 1: Build normal location baseline from sign-in logs
let user_normal_locations = SigninLogs
| where TimeGenerated > ago(90d)
| where ResultType == 0
| extend Country = tostring(LocationDetails.countryOrRegion)
| where isnotempty(Country)
| summarize NormalCountries = make_set(Country, 20) by UserPrincipalName;
// Step 2: Get breach-associated IP geolocations from SpyCloud
let breach_locations = SpyCloudBreachWatchlist_CL
| where TimeGenerated > ago({LOOKBACK}d)
| where isnotempty(ip_addresses)
| mv-expand ip = ip_addresses
| extend breach_ip = tostring(ip)
| where isnotempty(breach_ip)
| summarize
    BreachIPs       = make_set(breach_ip, 20),
    MaxSeverity     = max(severity),
    Exposures       = count(),
    InfectedDevices = dcount(infected_machine_id)
    by email;
// Step 3: Look for sign-ins from breach-associated IPs
SigninLogs
| where TimeGenerated > ago(14d)
| where ResultType == 0
| extend Country = tostring(LocationDetails.countryOrRegion),
         City = tostring(LocationDetails.city)
| where IPAddress in ((breach_locations | mv-expand ip = BreachIPs | project tostring(ip)))
        or UserPrincipalName in~ (breach_locations | project email)
| join kind=leftouter user_normal_locations on UserPrincipalName
| extend IsAnomalousLocation = NormalCountries !has Country
| where IsAnomalousLocation == true
| join kind=leftouter breach_locations on $left.UserPrincipalName == $right.email
| summarize
    AnomalousSignIns = count(),
    AnomalousCountries = make_set(Country, 10),
    AnomalousCities = make_set(City, 10),
    AnomalousIPs = make_set(IPAddress, 10),
    AppsAccessed = make_set(AppDisplayName, 10)
    by UserPrincipalName, MaxSeverity, Exposures,
       tostring(NormalCountries)
| order by AnomalousSignIns desc, MaxSeverity desc
"""

df_hunt8 = run_query(hunt8_kql)
display(df_hunt8)

In [ ]:
# Hunt 8 - Visualization: Geographic anomaly chart
if not df_hunt8.empty:
    # Explode anomalous countries for a country-level view
    country_data = df_hunt8.explode("AnomalousCountries")
    country_agg = (
        country_data
        .groupby("AnomalousCountries")
        .agg(Users=('UserPrincipalName', 'nunique'),
             TotalSignIns=('AnomalousSignIns', 'sum'))
        .reset_index()
        .sort_values("Users", ascending=False)
    )

    fig = px.bar(
        country_agg.head(15),
        x="AnomalousCountries",
        y="Users",
        color="TotalSignIns",
        color_continuous_scale="Reds",
        title="Hunt 8: Anomalous Sign-in Countries (not in user's normal baseline)",
        labels={
            "AnomalousCountries": "Country",
            "Users": "Affected Users",
            "TotalSignIns": "Sign-ins"
        }
    )
    fig.update_layout(height=500)
    fig.show()
else:
    print("No geographic anomalies detected.")

---
## Summary Dashboard

Aggregate key metrics from all hunts into a single at-a-glance summary for SOC leadership
and shift handoff reports.

In [ ]:
# Summary Dashboard - aggregate all hunt results
summary_kql = f"""
SpyCloudBreachWatchlist_CL
| where TimeGenerated > ago({LOOKBACK}d)
| summarize
    TotalRecords          = count(),
    UniqueEmails          = dcount(email),
    UniqueEmailDomains    = dcount(email_domain),
    PlaintextPasswords    = countif(isnotempty(password_plaintext)),
    AvgSeverity           = round(avg(severity), 1),
    MaxSeverity           = max(severity),
    CriticalRecords       = countif(severity >= 20),
    HighRecords           = countif(severity >= 15 and severity < 20),
    MediumRecords         = countif(severity >= 10 and severity < 15),
    InfectedDevices       = dcount(infected_machine_id),
    BreachSources         = dcount(source_id),
    TargetDomains         = dcount(target_domain),
    SessionTokenRecords   = countif(password_type has_any ('cookie', 'session', 'token')),
    OldestRecord          = min(spycloud_publish_date),
    NewestRecord          = max(spycloud_publish_date)
"""

df_summary = run_query(summary_kql)

if not df_summary.empty:
    s = df_summary.iloc[0]
    html = f"""
    <div style="font-family: sans-serif; padding: 20px;">
    <h2>SpyCloud Threat Hunting — Summary Dashboard</h2>
    <p>Lookback period: <strong>{LOOKBACK} days</strong> | Generated: {datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}</p>
    <table style="border-collapse: collapse; width: 100%; max-width: 900px;">
    <tr style="background:#1a237e; color:white;">
      <th style="padding:10px; text-align:left;">Metric</th>
      <th style="padding:10px; text-align:right;">Value</th>
    </tr>
    <tr><td style="padding:8px; border-bottom:1px solid #ddd;">Total Exposure Records</td>
        <td style="padding:8px; border-bottom:1px solid #ddd; text-align:right; font-weight:bold;">{s.TotalRecords:,.0f}</td></tr>
    <tr style="background:#f5f5f5;"><td style="padding:8px; border-bottom:1px solid #ddd;">Unique Exposed Users</td>
        <td style="padding:8px; border-bottom:1px solid #ddd; text-align:right; font-weight:bold;">{s.UniqueEmails:,.0f}</td></tr>
    <tr><td style="padding:8px; border-bottom:1px solid #ddd;">Plaintext Passwords</td>
        <td style="padding:8px; border-bottom:1px solid #ddd; text-align:right; font-weight:bold; color:#d32f2f;">{s.PlaintextPasswords:,.0f}</td></tr>
    <tr style="background:#f5f5f5;"><td style="padding:8px; border-bottom:1px solid #ddd;">Critical Severity (20+)</td>
        <td style="padding:8px; border-bottom:1px solid #ddd; text-align:right; font-weight:bold; color:#d32f2f;">{s.CriticalRecords:,.0f}</td></tr>
    <tr><td style="padding:8px; border-bottom:1px solid #ddd;">High Severity (15–19)</td>
        <td style="padding:8px; border-bottom:1px solid #ddd; text-align:right; font-weight:bold; color:#f57c00;">{s.HighRecords:,.0f}</td></tr>
    <tr style="background:#f5f5f5;"><td style="padding:8px; border-bottom:1px solid #ddd;">Medium Severity (10–14)</td>
        <td style="padding:8px; border-bottom:1px solid #ddd; text-align:right; font-weight:bold; color:#fbc02d;">{s.MediumRecords:,.0f}</td></tr>
    <tr><td style="padding:8px; border-bottom:1px solid #ddd;">Infected Devices (Infostealer)</td>
        <td style="padding:8px; border-bottom:1px solid #ddd; text-align:right; font-weight:bold;">{s.InfectedDevices:,.0f}</td></tr>
    <tr style="background:#f5f5f5;"><td style="padding:8px; border-bottom:1px solid #ddd;">Stolen Session Tokens</td>
        <td style="padding:8px; border-bottom:1px solid #ddd; text-align:right; font-weight:bold; color:#d32f2f;">{s.SessionTokenRecords:,.0f}</td></tr>
    <tr><td style="padding:8px; border-bottom:1px solid #ddd;">Distinct Breach Sources</td>
        <td style="padding:8px; border-bottom:1px solid #ddd; text-align:right; font-weight:bold;">{s.BreachSources:,.0f}</td></tr>
    <tr style="background:#f5f5f5;"><td style="padding:8px; border-bottom:1px solid #ddd;">Third-Party Domains</td>
        <td style="padding:8px; border-bottom:1px solid #ddd; text-align:right; font-weight:bold;">{s.TargetDomains:,.0f}</td></tr>
    <tr><td style="padding:8px; border-bottom:1px solid #ddd;">Average Severity</td>
        <td style="padding:8px; border-bottom:1px solid #ddd; text-align:right;">{s.AvgSeverity}</td></tr>
    <tr style="background:#f5f5f5;"><td style="padding:8px;">Data Range</td>
        <td style="padding:8px; text-align:right;">{s.OldestRecord} &mdash; {s.NewestRecord}</td></tr>
    </table>
    </div>
    """
    display(HTML(html))
else:
    print("No SpyCloud data found in the workspace.")

In [ ]:
# Summary - Hunt result counts
hunt_results = {
    "Hunt 1: Executive Accounts": len(df_hunt1) if 'df_hunt1' in dir() and not df_hunt1.empty else 0,
    "Hunt 2: Credential Stuffing": len(df_hunt2) if 'df_hunt2' in dir() and not df_hunt2.empty else 0,
    "Hunt 3: Lateral Movement": len(df_hunt3) if 'df_hunt3' in dir() and not df_hunt3.empty else 0,
    "Hunt 4: Exposure Timeline": len(df_hunt4) if 'df_hunt4' in dir() and not df_hunt4.empty else 0,
    "Hunt 5: Third-Party Risk": len(df_hunt5) if 'df_hunt5' in dir() and not df_hunt5.empty else 0,
    "Hunt 6: Password Reuse": len(df_hunt6) if 'df_hunt6' in dir() and not df_hunt6.empty else 0,
    "Hunt 7: Botnet Infections": len(df_hunt7) if 'df_hunt7' in dir() and not df_hunt7.empty else 0,
    "Hunt 8: Geo Anomalies": len(df_hunt8) if 'df_hunt8' in dir() and not df_hunt8.empty else 0,
}

fig = go.Figure(go.Bar(
    x=list(hunt_results.values()),
    y=list(hunt_results.keys()),
    orientation='h',
    marker_color=[
        '#d32f2f' if v > 0 else '#9e9e9e' for v in hunt_results.values()
    ]
))
fig.update_layout(
    title="Threat Hunt Results Summary — Rows Returned per Hunt",
    xaxis_title="Records Found",
    height=400,
    yaxis=dict(autorange="reversed")
)
fig.show()

total_findings = sum(hunt_results.values())
active_hunts = sum(1 for v in hunt_results.values() if v > 0)
display(Markdown(
    f"**Hunting complete.** {active_hunts}/8 hunts returned results "
    f"with {total_findings} total findings to investigate."
))